<a href="https://colab.research.google.com/github/07Akshaya/Statistical-Learning-e22019/blob/main/Assignment7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Task 1: Prior Belief Boundaries

#### Analytical Expectation

For a random variable following a Beta distribution $\Theta \sim \text{Beta}(\alpha, \beta)$, the analytical mean is given by:

$$E[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta}$$

Substituting $\alpha = 8$ and $\beta = 1.5$:

$$E[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.842105 \quad (84.21\%)$$

#### Engineering Rationale

*   **Physical Boundary Enforcement:** The Beta distribution's domain is strictly $(0, 1]$, which physically prevents non-sense values (such as negative stiffness or $>100\%$ efficiency).
*   **Prior Assumption of Structural Integrity:** With parameters $\alpha = 8$ and $\beta = 1.5$, the mode occurs at:

$$\text{Mode}(\Theta) = \frac{\alpha - 1}{\alpha + \beta - 2} = \frac{7}{7.5} \approx 0.9333$$

This heavily skews probability density toward pristine health ($\theta \approx 1.0$), reflecting high confidence in standard manufacturing quality while assigning near-zero prior density to severe degradation ($\theta \to 0$).

### Task 2: Structural Likelihood Formulation

#### Single Sensor Measurement Likelihood

Given the physical degradation model with multiplicative log-normal noise:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \quad \text{where } \epsilon_k \sim \mathcal{N}(0, \sigma^2)$$

Taking the natural logarithm of both sides:

$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k, \quad \epsilon_k \sim \mathcal{N}(0, \sigma^2)$$

$$\implies \ln(y_k) \sim \mathcal{N}\left(\ln(\theta \cdot K_{\text{nominal}}), \sigma^2\right)$$

Applying the change-of-variables transformation $f_{Y_k}(y_k) = f_{\ln Y_k}(\ln y_k) \cdot \left\vert{} \frac{d \ln y_k}{d y_k} \right\vert{} = f_{\ln Y_k}(\ln y_k) \cdot \frac{1}{y_k}$, the likelihood contribution $L(y_k \mid \theta)$ of observation $y_k$ conditional on $\theta$ is:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left( \ln y_k - \ln(\theta K_{\text{nominal}}) \right)^2}{2\sigma^2} \right)$$

#### Joint Likelihood Function

Assuming conditionally independent sensor noise terms given $\theta$, the joint likelihood for history $y^{(k)} = (y_1, y_2, \dots, y_k)^T$ is:

$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \left( \frac{1}{\sigma \sqrt{2\pi}} \right)^k \left( \prod_{i=1}^k \frac{1}{y_i} \right) \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left( \ln y_i - \ln(\theta K_{\text{nominal}}) \right)^2 \right)$$

### Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

#### Non-Conjugacy Justification

Conjugacy requires that the prior and posterior belong to the same family of probability distributions.

*   The **Beta prior** density kernel has an algebraic form: $\theta^{\alpha-1} (1-\theta)^{\beta-1}$.
*   The **Log-Normal likelihood** kernel in terms of $\theta$ is transcendental: $\exp\left(-\frac{(\ln y_k - \ln \theta - \ln K_{\text{nominal}})^2}{2\sigma^2}\right)$.

Multiplying these expressions yields a product that cannot be simplified algebraically into a standard parameterized family. The denominator normalizing integral $Z_k = \int_0^1 L(y_k \mid s) f(s) ds$ lacks a closed-form solution, necessitating numerical integration on a discrete grid.

#### Recursive Update Formula

The posterior density at inspection step $k$ updates recursively:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})}{\int_{0}^{1} L(y_k \mid s) \cdot f_{\Theta \mid Y^{(k-1)}}(s \mid y^{(k-1)}) \, ds}, \quad \theta \in (0, 1]$$

Or in unnormalized form:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

### Task 4: Running Point Estimates

The definite integral equations over the physical domain $(0, 1]$ are:

1.  **Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$):**

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid y^{(k)}] = \int_{0}^{1} \theta \cdot f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \, d\theta$$

2.  **Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$):**

$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)})$$

### Task 5: Algorithmic Grid Approximation and Normalization

To maintain the continuous state distribution computationally:

1.  **Domain Discretization:** Construct an array of $M$ equally spaced points over the operational interval $[0.01, 1.0]$:

$$\theta_m = 0.01 + (m-1)\Delta\theta, \quad \Delta\theta = \frac{1.0 - 0.01}{M-1}, \quad m = 1, 2, \dots, M$$

2.  **Prior Initialization:** Compute the initial density vector $P_0(\theta_m) = \text{Beta}(\theta_m; 8, 1.5)$ and normalize via the composite trapezoidal rule:

$$Z_0 = \text{trapezoid}(P_0, \theta), \quad P_0 \leftarrow \frac{P_0}{Z_0}$$

3.  **Sequential Processing Step $k$:** Upon observing sensor reading $y_k$:
*   Evaluate likelihood vector across the grid:

$$L(y_k \mid \theta_m) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln y_k - \ln(\theta_m K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

*   Element-wise update: $\tilde{P}_k(\theta_m) = P_{k-1}(\theta_m) \cdot L(y_k \mid \theta_m)$
*   Trapezoidal Normalization using `np.trapezoid`:

$$Z_k = \text{trapezoid}(\tilde{P}_k, \theta), \quad P_k(\theta_m) = \frac{\tilde{P}_k(\theta_m)}{Z_k}$$

4.  **Point Estimator Extraction:**
*   $\hat{\theta}_{\text{Bayes}}^{(k)} \approx \text{trapezoid}(\theta \odot P_k, \theta)$
*   $\hat{\theta}_{\text{MAP}}^{(k)} = \theta_{m^*}, \quad \text{where } m^* = \arg\max_m P_k(\theta_m)$

### Task 6: Python Implementation & Degradation Analysis

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set random seed for reproducibility
np.random.seed(24)

# Config & Parameters
theta_true = 0.68      # Hidden true stiffness factor (32% stiffness loss)
K_nominal = 50.0       # Pristine baseline stiffness (kN/mm)
sigma = 0.15           # Sensor log-space noise std dev
n_steps = 15           # Number of sequential inspection readings

# Helper for trapezoidal integration (handles NumPy version differences)
trapz = getattr(np, 'trapezoid', getattr(np, 'trapz', None))

# 1. Define bounded grid over [0.01, 1.0]
theta_grid = np.linspace(0.01, 1.0, 1000)

# 2. Initialize Prior: Beta(8, 1.5)
prior_pdf = stats.beta.pdf(theta_grid, a=8.0, b=1.5)
prior_pdf /= trapz(prior_pdf, theta_grid)

# Initial Point Estimators at Step 0
initial_bayes = trapz(theta_grid * prior_pdf, theta_grid)
initial_map = theta_grid[np.argmax(prior_pdf)]

# Data collection arrays across steps k=0..15
bayes_timeline = [initial_bayes]
map_timeline = [initial_map]
milestones = [0, 1, 2, 5, 10, 15]
milestone_posteriors = {0: prior_pdf.copy()}

current_posterior = prior_pdf.copy()

# 3. Sequential Monitoring Loop
for k in range(1, n_steps + 1):
    # Simulate log-normal noise observation y_k = theta_true * K_nom * exp(N(0, sigma^2))
    eps = np.random.normal(0, sigma)
    y_k = theta_true * K_nominal * np.exp(eps)

    # Log-Normal Likelihood curve across grid
    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    # Bayesian Update & Trapezoidal Normalization
    unnorm_posterior = current_posterior * likelihood
    Z_k = trapz(unnorm_posterior, theta_grid)
    current_posterior = unnorm_posterior / Z_k

    # Extract Point Estimates
    e_bayes = trapz(theta_grid * current_posterior, theta_grid)
    e_map = theta_grid[np.argmax(current_posterior)]

    bayes_timeline.append(e_bayes)
    map_timeline.append(e_map)

    if k in milestones:
        milestone_posteriors[k] = current_posterior.copy()

# 4. Generate Two Plots using Plotly Subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "Posterior Density Curves Across Inspection Milestones",
        "Convergence of Point Estimators to True Degradation State (θ_true = 0.68)"
    ),
    vertical_spacing=0.12
)

# Plot 1: Milestone Posterior Profiles
colors = ['gray', '#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#d62728']
for idx, k_val in enumerate(milestones):
    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=milestone_posteriors[k_val],
            mode='lines',
            name=f"Step k={k_val}",
            line=dict(width=2.5 if k_val in [0, 15] else 1.5, color=colors[idx])
        ),
        row=1, col=1
    )

fig.add_vline(
    x=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text="True State (θ=0.68)", row=1, col=1
)

# Plot 2: Convergence Timeline
steps_array = list(range(0, n_steps + 1))
fig.add_trace(
    go.Scatter(
        x=steps_array, y=bayes_timeline, mode='lines+markers',
        name='Bayes Posterior Mean', line=dict(color='blue', width=2), marker=dict(size=6)
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=steps_array, y=map_timeline, mode='lines+markers',
        name='MAP Estimate', line=dict(color='green', width=2, dash='dot'), marker=dict(size=6)
    ),
    row=2, col=1
)
fig.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text="True Degradation Threshold (0.68)", row=2, col=1
)

# Layout Formatting
fig.update_layout(
    height=800,
    title_text="Structural Health Monitoring: Bounded Bayesian Grid Updating",
    template="plotly_white",
    hovermode="x unified"
)
fig.update_xaxes(title_text="Stiffness Efficiency Factor (θ)", row=1, col=1)
fig.update_yaxes(title_text="Probability Density", row=1, col=1)
fig.update_xaxes(title_text="Inspection Step (k)", dtick=1, row=2, col=1)
fig.update_yaxes(title_text="Estimated Efficiency (θ)", row=2, col=1)

fig.show()


### Degradation Convergence Analysis

1.  **Overcoming the Optimistic Prior ($k \approx 3\text{--}5$ steps):**
*   At step $k=0$, the prior mean assumes an optimistic baseline condition ($\hat{\theta}_{\text{Bayes}}^{(0)} \approx 0.8421$, $\hat{\theta}_{\text{MAP}}^{(0)} \approx 0.9333$).
*   By step $k=2$, consecutive noisy observations centered around $y_k \approx 34 \text{ kN/mm}$ shift the posterior probability mass sharply to the left.
*   By step $k=4\text{--}5$, both estimators cross below $0.70$, successfully overriding the biased initial prior and isolating the true damage level ($\theta_{\text{true}} = 0.68$).

2.  **Narrowing of Density Curves & Safety Implications:**
*   **Variance Attenuation:** As data accumulates ($k \to 15$), the spread (variance) of the density curve decreases substantially, narrowing into a sharp peak centered around $\theta = 0.68$.
*   **Structural Safety Thresholds:** In structural integrity management, a narrow posterior density implies high certainty. Engineers can confidently declare a **32% structural stiffness loss** without risking false alarm triggers or missing critical damage. The bounded framework strictly guarantees no unphysical states ($\theta < 0$ or $\theta > 1$) enter risk evaluation calculations.